In [1]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import zmq

from bcs import BCSz


In [2]:
server = BCSz.BCSServer()

In [3]:
conn = await server.connect(addr="localhost", port=5577)

Server Public Key b'3=KZYI@pOlgBr1svn7=c7$Xh+f>U^t9S58jP]h&a'


In [4]:
await server.start_instrument_acquire("Axis Photonique", acq_time_s=.002)

{'success': True,
 'error description': 'no error',
 'log?': True,
 'elapsed_s': 0.593579499996849,
 'API_delta_t': 0.6057074069976807}

In [5]:
start = perf_counter()

img = await server.get_instrument_acquired2d_string("Axis Photonique")
final_acq = perf_counter() - start

print(final_acq)

height, width, s = img["Height"], img["Width"], img["Data"]
# extract the array from the string
parts = [x for x in s.strip().split(",") if x]
array = np.array(parts, dtype=np.int32).reshape(height, width)


final_acq = perf_counter() - start

print(final_acq)

6.463318299967796
11.787383999908343


In [6]:
start = perf_counter()
img = await server.get_instrument_acquired2d_base85("Axis Photonique")
final_acq = perf_counter() - start

print(final_acq)

height, width, s = img["Height"], img["Width"], img["Data"]

1.1821155999787152


In [7]:
t0 = perf_counter()
raw = zmq.utils.z85.decode(s) # no need to encode the string first this saves a second
t1 = perf_counter()
arr = np.frombuffer(raw, dtype=np.uint8)
t2 = perf_counter()
img = arr.reshape(height, width)
t3 = perf_counter()


In [8]:
print(f"z85 decode: {t1 - t0: .2g}s")
print(f"frombuffer: {t2 - t1: .2g}s")
print(f"reshape: {t3 - t2: .2g}s")

z85 decode:  5.8s
frombuffer:  0.00016s
reshape:  9.1e-05s


In [9]:
_Z85_ALPHABET = b'0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ.-:+=^!/*?&<>()[]{}@%$#'
_Z85_DEC = np.full(255, 0, np.uint64)
for i, c in enumerate(_Z85_ALPHABET):
    _Z85_DEC[c] = i
_Z85_POW = np.array([85**i for i in range(4, -1, -1)], dtype=np.uint64)

def z85decode_vectorized(data: str | bytes) -> np.ndarray:
    b=np.frombuffer(data if isinstance(data, bytes) else data.encode(), dtype=np.uint8)
    digits = _Z85_DEC[b].reshape(-1, 5)
    values = (digits * _Z85_POW).sum(axis=1).astype(np.uint32)
    return values.astype(">u4").view(np.uint8)

In [10]:
t0_2 = perf_counter()
raw = z85decode_vectorized(s)
t1_2 = perf_counter()
arr = np.frombuffer(raw, dtype=np.uint8)
t2_2 = perf_counter()
img = arr.reshape(height, width)
t3_2 = perf_counter()

In [11]:
print(f"z85_vectorized: {t1_2 - t0_2: .2g}s")
print(f"frombuffer: {t2_2 - t1_2: .2g}s")
print(f"reshape: {t3_2 - t2_2: .2g}s")

z85_vectorized:  0.38s
frombuffer:  0.00015s
reshape:  0.0016s


In [12]:
# a little bit of rust magic 
from bcs_rs import decode_z85_parallel, decode_z85 

In [13]:
t0_3 = perf_counter()
raw = decode_z85(s)
t1_3 = perf_counter()
t0_4 = perf_counter()
raw = decode_z85_parallel(s)
t1_4 = perf_counter()

In [14]:
print(f"decode_z85: {t1_3 - t0_3: .2g}s")
print(f"decode_z85_parallel: {t1_4 - t0_4: .2g}s")

decode_z85:  0.062s
decode_z85_parallel:  0.043s


In [ ]:
array = np.frombuffer(zmq.utils.z85.decode(s), dtype=np.uint8).reshape(height, width)

acq_zmq = perf_counter() - start

print(acq_zmq)

array_2 = _parse_payload(img)

In [17]:
import json

async def bcs_request(self, command_name, param_dict, debugging=False):
    t0 = perf_counter()
    param_dict = dict(param_dict)
    param_dict["command"] = command_name
    param_dict["_unused"] = "_unused"
    if "self" in param_dict:
        del param_dict["self"]
    request_bytes = json.dumps(param_dict).encode()
    t_serialize = perf_counter()

    await self._zmq_socket.send(request_bytes)
    t_send = perf_counter()

    response_bytes = await self._zmq_socket.recv()
    t_recv = perf_counter()

    response_dict = json.loads(response_bytes)
    t_deserialize = perf_counter()

    response_dict["API_delta_t"] = t_deserialize - t0
    response_dict["_profile"] = {
        "serialize_ms": (t_serialize - t0),
        "send_ms": (t_send - t_serialize),
        "recv_ms": (t_recv - t_send),
        "deserialize_ms": (t_deserialize - t_recv),
        "total_s": (t_deserialize - t0),
    }
    if debugging:
        print(f"API {command_name} profile: {response_dict['_profile']}")
    return response_dict

# test using the "GetInstrumentAcquired2DBase85" endpoint with "Axis Photonique"
payload = await bcs_request(server, "GetInstrumentAcquired2DBase85", {"name": "Axis Photonique"})
profile = payload["_profile"]
sum_sec = 0
for k, v in profile.items():
    sum_sec += v
    print(f"{k}: {v:.2g}")

longest_step_name = max([k for k in profile if k != "total_s"], key=lambda k: profile[k])
longest_step_time = profile[longest_step_name]
print(f"Longest step: {longest_step_name} = {longest_step_time:.2f} s")


serialize_ms: 2.1e-05
send_ms: 0.00016
recv_ms: 1.2
deserialize_ms: 0.068
total_s: 1.3
Longest step: recv_ms = 1.21 s


In [18]:
from bcs_rs import BCSServer

In [19]:
server = BCSServer()

conn = await server.connect(addr="localhost", port=5577)


RuntimeError: ZMQ: Not supported

In [ ]:

plt.imshow(array)
plt.show()

final_acq = perf_counter() - start

print(final_acq)